# 시선 정확도 향상 학습 (GPU)

기존 `train_gaze_aihub.py` 대비:
- **강한 augmentation**: RandomPerspective, 더 넓은 ColorJitter, Grayscale 등
- **조금 더 큰 regression head**: 128→256 hidden
- **동일 데이터**: `runs/aihub_cache` (추가 준비 없이 바로 실행)

실행 전: 노트북 커널이 **GPU** 사용 중인지 확인하세요.

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from pathlib import Path
from PIL import Image

os.environ['PYTHONIOENCODING'] = 'utf-8'

# 노트북/스크립트 위치 기준 (Jupyter는 보통 노트북 폴더가 cwd)
NB_DIR = Path('.').resolve()
BASE = NB_DIR
for candidate in [NB_DIR, NB_DIR.parent, Path(__file__).resolve().parent if '__file__' in dir() else NB_DIR]:
    try:
        if (candidate / 'runs' / 'aihub_cache' / 'train_eyes.npy').exists():
            BASE = candidate
            break
        if (candidate / 'GazeCapture' / 'MPIIGAZE' / 'runs' / 'aihub_cache' / 'train_eyes.npy').exists():
            BASE = candidate / 'GazeCapture' / 'MPIIGAZE'
            break
    except Exception:
        pass

CACHE_DIR = BASE / 'runs' / 'aihub_cache'
RUNS_DIR  = BASE / 'runs' / 'gaze_aihub'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS        = 70
PHASE1_EPOCHS = 25
LR_HEAD       = 1e-3
LR_FINETUNE   = 5e-5
WEIGHT_DECAY  = 1e-3
BATCH         = 128
IMG_SIZE      = 64
SCREEN_W      = 2560
SCREEN_H      = 1440
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'BASE: {BASE}')
print(f'Cache: {CACHE_DIR}  exists={CACHE_DIR.exists()}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE: C:\Users\SSAFY\Desktop\gaze-capture\GazeCapture\MPIIGAZE
Cache: C:\Users\SSAFY\Desktop\gaze-capture\GazeCapture\MPIIGAZE\runs\aihub_cache  exists=True
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
class GazeScreenDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        if self.transform:
            img = self.transform(img)
        return img, torch.from_numpy(self.labels[idx])

In [3]:
# 정확도 향상: regression head 128→256, dropout 유지
class GazeEstimatorBoost(nn.Module):
    def __init__(self, dropout: float = 0.5, hidden: int = 256):
        super().__init__()
        bb  = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        inf = bb.classifier[0].in_features
        bb.classifier = nn.Sequential(
            nn.Linear(inf, hidden),
            nn.Hardswish(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden, 64),
            nn.Hardswish(),
            nn.Dropout(p=dropout * 0.6),
            nn.Linear(64, 2),
            nn.Sigmoid(),
        )
        self.net = bb

    def freeze_backbone(self):
        for p in self.net.features.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.net.features.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.net(x)

In [4]:
def pixel_error(pred: torch.Tensor, tgt: torch.Tensor) -> float:
    dx = (pred[:, 0] - tgt[:, 0]) * SCREEN_W
    dy = (pred[:, 1] - tgt[:, 1]) * SCREEN_H
    return torch.sqrt(dx * dx + dy * dy).mean().item()

In [5]:
# 데이터 로드
tr_imgs = np.load(str(CACHE_DIR / 'train_eyes.npy'))
tr_lbl  = np.load(str(CACHE_DIR / 'train_labels.npy'))
va_imgs = np.load(str(CACHE_DIR / 'val_eyes.npy'))
va_lbl  = np.load(str(CACHE_DIR / 'val_labels.npy'))
print(f'Train {len(tr_lbl):,}  Val {len(va_lbl):,}')

Train 26,058  Val 30,736


In [6]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# 강한 augmentation (정확도 향상용)
tr_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomAffine(degrees=8, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
va_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

tr_ds = GazeScreenDataset(tr_imgs, tr_lbl, tr_tf)
va_ds = GazeScreenDataset(va_imgs, va_lbl, va_tf)
tr_ld = DataLoader(tr_ds, BATCH, shuffle=True,  num_workers=0, pin_memory=True)
va_ld = DataLoader(va_ds, BATCH, shuffle=False, num_workers=0, pin_memory=True)

In [7]:
# run 폴더 자동 증가
idx = 1
while (RUNS_DIR / f'run{idx}').exists():
    idx += 1
run_dir = RUNS_DIR / f'run{idx}'
run_dir.mkdir()
print(f'Run: run{idx}')

model = GazeEstimatorBoost(dropout=0.5, hidden=256).to(DEVICE)
model.freeze_backbone()
criterion = nn.SmoothL1Loss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=LR_HEAD * 0.1)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

best_px   = float('inf')
log_lines = ['epoch,train_loss,val_loss,val_px_err,lr,phase']
phase     = 1

Run: run4


In [8]:
# Phase 1 학습 (백본 동결)
for ep in range(1, PHASE1_EPOCHS + 1):
    model.train()
    tl = 0.0
    for imgs, labels in tr_ld:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                pred = model(imgs)
                loss = criterion(pred, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(imgs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        tl += loss.item() * imgs.size(0)
    tl /= len(tr_ds)

    model.eval()
    vl = 0.0
    ap, at = [], []
    with torch.no_grad():
        for imgs, labels in va_ld:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            pred = model(imgs)
            loss = criterion(pred, labels)
            vl += loss.item() * imgs.size(0)
            ap.append(pred.cpu())
            at.append(labels.cpu())
    vl /= len(va_ds)
    vp = pixel_error(torch.cat(ap), torch.cat(at))
    lr_now = optimizer.param_groups[0]['lr']
    scheduler.step()

    log_lines.append(f'{ep},{tl:.6f},{vl:.6f},{vp:.2f},{lr_now:.8f},{phase}')
    if vp < best_px:
        best_px = vp
        torch.save({'epoch': ep, 'model_state': model.state_dict(), 'val_px': vp}, str(run_dir / 'best.pt'))

    if ep % 5 == 0 or ep == 1:
        print(f'Phase1  Epoch {ep:3d}  TrainLoss {tl:.5f}  ValLoss {vl:.5f}  PxErr {vp:.1f}  LR {lr_now:.2e}')

print(f'Phase 1 완료. Best Val Px Error: {best_px:.1f} px')

Phase1  Epoch   1  TrainLoss 0.03739  ValLoss 0.03687  PxErr 740.5  LR 1.00e-03


Phase1  Epoch   5  TrainLoss 0.03686  ValLoss 0.03772  PxErr 752.0  LR 9.44e-04


Phase1  Epoch  10  TrainLoss 0.03675  ValLoss 0.03791  PxErr 751.3  LR 7.42e-04


Phase1  Epoch  15  TrainLoss 0.03668  ValLoss 0.03754  PxErr 746.8  LR 4.66e-04


Phase1  Epoch  20  TrainLoss 0.03662  ValLoss 0.03738  PxErr 744.1  LR 2.22e-04


Phase1  Epoch  25  TrainLoss 0.03654  ValLoss 0.03775  PxErr 745.0  LR 1.04e-04
Phase 1 완료. Best Val Px Error: 724.1 px


In [9]:
# Phase 2: 전체 fine-tune
model.unfreeze_backbone()
optimizer = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - PHASE1_EPOCHS, eta_min=1e-7
)
if scaler:
    scaler = torch.amp.GradScaler('cuda')
phase = 2
last_best_epoch = PHASE1_EPOCHS  # early stopping용
print('Phase 2: full fine-tune 시작')

Phase 2: full fine-tune 시작


In [10]:
for ep in range(PHASE1_EPOCHS + 1, EPOCHS + 1):
    model.train()
    tl = 0.0
    for imgs, labels in tr_ld:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                pred = model(imgs)
                loss = criterion(pred, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(imgs)
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()
        tl += loss.item() * imgs.size(0)
    tl /= len(tr_ds)

    model.eval()
    vl = 0.0
    ap, at = [], []
    with torch.no_grad():
        for imgs, labels in va_ld:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            pred = model(imgs)
            loss = criterion(pred, labels)
            vl += loss.item() * imgs.size(0)
            ap.append(pred.cpu())
            at.append(labels.cpu())
    vl /= len(va_ds)
    vp = pixel_error(torch.cat(ap), torch.cat(at))
    lr_now = optimizer.param_groups[0]['lr']
    scheduler.step()

    log_lines.append(f'{ep},{tl:.6f},{vl:.6f},{vp:.2f},{lr_now:.8f},{phase}')
    if vp < best_px:
        best_px = vp
        last_best_epoch = ep
        torch.save({'epoch': ep, 'model_state': model.state_dict(), 'val_px': vp}, str(run_dir / 'best.pt'))

    if (ep - PHASE1_EPOCHS) % 5 == 0 or ep == PHASE1_EPOCHS + 1:
        print(f'Phase2  Epoch {ep:3d}  TrainLoss {tl:.5f}  ValLoss {vl:.5f}  PxErr {vp:.1f}  LR {lr_now:.2e}')

    # Early stopping: 15 에포크 동안 개선 없으면 종료
    if ep - last_best_epoch >= 15:
        print(f'Early stopping at epoch {ep} (no improvement for 15 epochs)')
        break

print(f'Best Val Pixel Error: {best_px:.1f} px')
(run_dir / 'results.csv').write_text('\n'.join(log_lines), encoding='utf-8')

Phase2  Epoch  26  TrainLoss 0.03649  ValLoss 0.03820  PxErr 745.7  LR 5.00e-05


Phase2  Epoch  30  TrainLoss 0.03617  ValLoss 0.03927  PxErr 754.3  LR 4.90e-05


Phase2  Epoch  35  TrainLoss 0.03563  ValLoss 0.04092  PxErr 763.0  LR 4.52e-05


Phase2  Epoch  40  TrainLoss 0.03530  ValLoss 0.04150  PxErr 769.8  LR 3.90e-05
Early stopping at epoch 40 (no improvement for 15 epochs)
Best Val Pixel Error: 724.1 px


1676

In [11]:
# ONNX 내보내기 (웹 데모용)
ckpt = torch.load(str(run_dir / 'best.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
onnx_path = str(run_dir / 'gaze_screen.onnx')
torch.onnx.export(
    model, dummy, onnx_path, opset_version=17,
    input_names=['eye_image'], output_names=['screen_coords'],
    dynamic_axes={'eye_image': {0: 'batch'}, 'screen_coords': {0: 'batch'}},
)
print(f'ONNX 저장: {onnx_path}')
print(f'웹 데모에 적용: 위 파일을 web_demo/gaze_screen.onnx 로 복사')

ONNX 저장: C:\Users\SSAFY\Desktop\gaze-capture\GazeCapture\MPIIGAZE\runs\gaze_aihub\run4\gaze_screen.onnx
웹 데모에 적용: 위 파일을 web_demo/gaze_screen.onnx 로 복사
